In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# Importing Libraries
## - torch for Tensor operations and model training
## - torch.nn for Neural network layers and loss functions
## - Dataset, DataLoader	for Organizing examples into batches
## - AutoTokenizer, AutoModel	for Loading DeBERTa's tokenizer and pretrained backbone
## - pandas	for Reading and manipulating CSV data
## - numpy	for Numerical operations and prediction arrays
## - sklearn.metrics	for Evaluating classification performance
## - typing	for Type hints for functions and classes
## - tqdm	for Progress bars during processing

In [ ]:
import torch
import torch.nn  as nn
from torch.utils.data import Dataset,DataLoader
from transformers import AutoTokenizer, AutoModel

import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score,f1_score,recall_score,precision_score
from typing import Dict,List,Optional
from tqdm import tqdm

# Loading Data

In [ ]:
train = pd.read_csv("/content/drive/MyDrive/datasets/2025-sep-dl-gen-ai-project/train.csv")
test = pd.read_csv("/content/drive/MyDrive/datasets/2025-sep-dl-gen-ai-project/test.csv")
sub = pd.read_csv("/content/drive/MyDrive/datasets/2025-sep-dl-gen-ai-project/sample_submission.csv")

# create a training and validation set
 The original training data is split into two subsets. train_df is used for model training, while val_df is held out for validation.
- test_size=0.2 reserves 20% of the rows for validation.
- random_state=42 makes the split reproducible when the input data and settings remain unchanged.
The validation subset is used to monitor model performance and tune prediction thresholds; it is not used for gradient updates in the training loop.

In [ ]:
from sklearn.model_selection import train_test_split
train_df,val_df = train_test_split(train,test_size=0.2,random_state = 42)

# Load the pretrained DeBERTa tokenizer
This cell selects the pretrained microsoft/deberta-v3-base model family and loads its matching tokenizer.
The tokenizer converts raw text into token IDs and an attention mask that the transformer can process. The same tokenizer should be used consistently for training, validation, and test inference.

In [ ]:
model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

# Prepare text and labels with a custom PyTorch Dataset
EmotionDataset adapts the DataFrame to the format expected by PyTorch:
- Reads the text from the text column.
- Collects the five target columns: anger, joy, fear, sadness, and surprise.
- Tokenizes each text with padding and truncation to max_len=256.
- Returns input_ids, attention_mask, and floating-point multi-label targets.
Padding makes each encoded example the configured length, while truncation cuts off tokens beyond that length. The final lines create training and validation datasets and inspect the shape of one example.

In [ ]:
class EmotionDataset(Dataset):
  def __init__(self,df:pd.DataFrame,tokenizer,max_len=256):
    self.texts = df['text'].to_list()
    self.labels = df[['anger','joy','fear','sadness','surprise']].values
    self.tokenizer = tokenizer
    self.max_len = max_len

  def __len__(self):
    return len(self.texts)

  def __getitem__(self,idx) -> Dict[str,torch.tensor]:
    text = str(self.texts[idx])
    labels = self.labels[idx]

    encoding = self.tokenizer(
        text,
        padding = 'max_length',
        truncation = True,
        max_length = self.max_len,
        return_tensors = 'pt'
    )
    return {
        "input_ids":encoding['input_ids'].flatten(),
        "attention_mask":encoding['attention_mask'].flatten(),
        "labels":torch.tensor(labels,dtype = torch.float)
    }

train_dataset = EmotionDataset(train_df,tokenizer)
val_dataset = EmotionDataset(val_df,tokenizer)

print(f"len of train dataset:{len(train_dataset)}")
print(f"len of val dataset:{len(val_dataset)}")
print("\nSample from train dataset")
sample = train_dataset[0]
for key,value in sample.items():
  print(f"{key}:{value.shape}")

len of train dataset:5461
len of val dataset:1366

Sample from train dataset
input_ids:torch.Size([256])
attention_mask:torch.Size([256])
labels:torch.Size([5])


# Collate examples into batches
The custom DataCollator stacks the tensors returned by the Dataset into batch tensors for the model.
Each batch contains:
- input_ids: token IDs for all examples in the batch.
- attention_mask: indicators for real tokens versus padding.
- labels: the five target values for each example.
This collator is passed to the DataLoaders in the next section.

In [ ]:
# DataCollator
from dataclasses import dataclass

@dataclass
class DataCollator:
    def __call__(self, batch: List[Dict[str, torch.tensor]]) -> Dict[str, torch.tensor]:
        input_ids = torch.stack([item['input_ids'] for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        labels = torch.stack([item['labels'] for item in batch])

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

collator = DataCollator()
print("Data collator initialized successfully!")

Data collator initialized successfully!


# The DataLoaders provide batches of examples to the training and validation loops.
- BATCH_SIZE = 16 sets the number of examples in a typical batch.
- Training uses shuffle=True so the example order is randomized each epoch.
- Validation uses shuffle=False to keep its order stable.
- collate_fn=collator uses the custom batching logic defined above.
These loaders separate data preparation from the model-training code.

In [ ]:
BATCH_SIZE = 16
train_loader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle = True,
    collate_fn = collator,
    num_workers = 2
)
val_loader = DataLoader(
    val_dataset,
    batch_size = BATCH_SIZE,
    shuffle = False,
    collate_fn = collator,
    num_workers = 2
)

#Define the DeBERTa emotion classifier
EmotionClassifier combines a pretrained DeBERTa backbone with a task-specific classification head.
1. AutoModel loads the transformer backbone.
2. last_hidden_state[:, 0, :] selects the contextual representation at the first token position.
3. Dropout is applied to that representation during training to help regularize the classifier.
4. A linear layer maps the representation to five raw logits, one for each emotion.
Because this is a multi-label task, the model returns logits rather than a single softmax class. The cell also selects CUDA when available, moves the model to the chosen device, and prints model and parameter information.

In [ ]:
class EmotionClassifier(nn.Module):
  def __init__(self,model_name:str,num_labels:int=5,dropout:float=0.3):
    super(EmotionClassifier,self).__init__()
    self.transformer = AutoModel.from_pretrained(model_name,dtype=torch.float32)
    self.dropout = nn.Dropout(dropout)
    self.classifier = nn.Linear(self.transformer.config.hidden_size,num_labels)

  def forward(self,input_ids:torch.tensor,attention_mask:torch.tensor) -> torch.tensor:
    output = self.transformer(input_ids = input_ids,attention_mask = attention_mask)
    pooled_output = output.last_hidden_state[:,0,:]
    pooled_output = self.dropout(pooled_output)
    logits = self.classifier(pooled_output)

    return logits

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmotionClassifier(model_name,num_labels=5)
model = model.to(device)


print(f"model loaded to device:{device}")
print(f"\nModel Architecture")
print(model)
print(f"total params:{sum(p.numel() for p in model.parameters()):,}")
print (f'total trainable parameters:{sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model loaded to device:cuda

Model Architecture
EmotionClassifier(
  (transformer): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=Tru

#The Trainer class organizes the training and validation loops.
#Training
For each batch, it moves inputs and labels to the selected device, clears old gradients, runs the model, calculates loss, backpropagates gradients, and updates parameters with the optimizer.
#Validation
The model is switched to evaluation mode and predictions are calculated without gradient tracking. Sigmoid converts logits into independent emotion scores; a default threshold of 0.5 is used for the initial binary predictions. The validation loop calculates loss and Macro F1.
The class also stores loss history so training and validation behavior can be monitored across epochs.

In [ ]:
class Trainer:
  def __init__(self, train_loader, val_loader, model, optimizer, criterion, device):
    self.train_loader = train_loader
    self.val_loader = val_loader
    self.model = model
    self.optimizer = optimizer
    self.criterion = criterion
    self.device = device
    self.history = {'train_loss':[], 'val_loss':[]}

  def train_epoch(self) -> float:
    self.model.train()
    total_loss = 0
    for batch in tqdm(self.train_loader, desc="Training"):
      input_ids = batch['input_ids'].to(self.device)
      attention_mask = batch['attention_mask'].to(self.device)
      labels = batch['labels'].to(self.device)

      self.optimizer.zero_grad()
      logits = self.model(input_ids=input_ids, attention_mask=attention_mask)
      loss = self.criterion(logits, labels)
      loss.backward()
      self.optimizer.step()
      total_loss += loss.item()
    avg_loss = total_loss / len(self.train_loader)
    return avg_loss

  def val_epoch(self) -> tuple:
    self.model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
      for batch in tqdm(self.val_loader, desc="Validation"):
        input_ids = batch['input_ids'].to(self.device)
        attention_mask = batch['attention_mask'].to(self.device)
        labels = batch['labels'].to(self.device)

        logits = self.model(input_ids=input_ids, attention_mask=attention_mask)
        loss = self.criterion(logits, labels)
        total_loss += loss.item()

        preds = torch.sigmoid(logits) > 0.5
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    avg_loss = total_loss / len(self.val_loader)
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, f1

  def train(self, num_epochs: int):
    print("Training Started....")
    for epoch in range(num_epochs):
      print(f"\nEpoch: {epoch+1}/{num_epochs}")
      train_loss = self.train_epoch()
      val_loss, f1 = self.val_epoch()
      self.history['train_loss'].append(train_loss)
      self.history['val_loss'].append(val_loss)
      print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val F1 Score: {f1:.4f}")

    best_f1 = 0.0
    if f1 > best_f1:
      best_f1 = f1
      torch.save(self.model.state_dict(), 'best_model.pth')
      print("Best model saved!")
    print("\nTraining completed!")

#Configure the loss, optimizer, and training run
This cell configures the optimization process:
- Loss: BCEWithLogitsLoss, suitable for independent binary targets in multi-label classification. It expects raw logits, so sigmoid should not be applied before calculating this loss.
- Optimizer: AdamW with learning rate 1e-5 and weight decay 0.01.
- Epochs: the model is trained for five passes through the training data.
The Trainer instance runs the training and validation loops. Review the printed metrics to understand how performance changes across epochs.

In [ ]:

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr= 1e-5, weight_decay=0.01)

NUM_EPOCHS = 5
trainer = Trainer(train_loader, val_loader, model, optimizer, criterion, device)
trainer.train(NUM_EPOCHS)

Training Started....

Epoch: 1/5


Validation: 100%|██████████| 86/86 [00:28<00:00,  3.05it/s]


Train Loss: 0.4736 | Val Loss: 0.3483 | Val F1 Score: 0.5984

Epoch: 2/5


Validation: 100%|██████████| 86/86 [00:28<00:00,  3.04it/s]


Train Loss: 0.3148 | Val Loss: 0.2863 | Val F1 Score: 0.7741

Epoch: 3/5


...

Train Loss: 0.2370 | Val Loss: 0.2712 | Val F1 Score: 0.7880

Epoch: 4/5


Validation: 100%|██████████| 86/86 [00:28<00:00,  3.04it/s]


Train Loss: 0.1754 | Val Loss: 0.2435 | Val F1 Score: 0.8240

Epoch: 5/5


Validation: 100%|██████████| 86/86 [00:28<00:00,  3.06it/s]


Train Loss: 0.1302 | Val Loss: 0.2449 | Val F1 Score: 0.8316
Best model saved!

Training completed!


#Generate validation probabilities
get_probabilities runs the model in evaluation mode and gathers sigmoid scores for each emotion. When has_labels=True, it also collects the corresponding ground-truth labels.
These validation probabilities are used in the next cells to search for a suitable threshold for each emotion. The function also supports inference without labels, which is useful for the test set.

In [ ]:
emotion_names = [
    "anger",
    "joy",
    "fear",
    "sadness",
    "surprise"
]

def get_probabilities(model, loader, device, has_labels=True):
    model.eval()
    all_probs = []
    all_labels = [] if has_labels else None
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = model(input_ids=input_ids,attention_mask=attention_mask)
            probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu().numpy())
            if has_labels:
                all_labels.append(batch["labels"].cpu().numpy())
    all_probs = np.vstack(all_probs)
    if has_labels:
        all_labels = np.vstack(all_labels)
        return all_probs, all_labels
    return all_probs



#Collect predictions for the validation set
This cell calls get_probabilities on the validation DataLoader and stores:
- val_probs: predicted sigmoid scores for each of the five emotions.
- val_labels: ground-truth multi-label targets.
Keeping the scores allows threshold tuning to be performed without rerunning the transformer for every threshold candidate.

In [ ]:
val_probs, val_labels = get_probabilities(
    model,
    val_loader,
    device,
    has_labels=True
)

Predicting: 100%|██████████| 86/86 [00:29<00:00,  2.94it/s]


#Tune a separate threshold for each emotion
A default threshold of 0.5 is not always optimal for every label. This cell searches thresholds from 0.10 through 0.90 in increments of 0.01.
For each emotion, it selects the threshold that gives the highest validation F1 for that label and stores it in best_thresholds. These thresholds are selected using the validation labels, so the resulting validation score is a tuning result rather than an independent test score.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.01)
best_thresholds = {}
print("\nBest Thresholds")
print("=" * 60)

for i, emotion in enumerate(emotion_names):
    best_f1 = 0
    best_threshold = 0.5
    for threshold in thresholds:
        preds = (val_probs[:, i] >= threshold).astype(int)
        f1 = f1_score(val_labels[:, i],preds,zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    best_thresholds[emotion] = best_threshold
    print(
        f"{emotion:10s} | "
        f"Threshold: {best_threshold:.2f} | "
        f"F1: {best_f1:.4f}"
    )


Best Thresholds
anger      | Threshold: 0.32 | F1: 0.7908
joy        | Threshold: 0.27 | F1: 0.8446
fear       | Threshold: 0.73 | F1: 0.8914
sadness    | Threshold: 0.22 | F1: 0.8581
surprise   | Threshold: 0.46 | F1: 0.8438


#Evaluate validation Macro F1 with tuned thresholds
The selected threshold for each emotion is applied to the validation probabilities to create binary predictions.
The cell then calculates Macro F1, which computes F1 for each label and averages the label scores equally. This gives each of the five emotions equal weight in the overall metric.

In [ ]:
optimized_val_preds = np.zeros_like(val_probs,dtype=int)
for i, emotion in enumerate(emotion_names):
    optimized_val_preds[:, i] = (val_probs[:, i] >= best_thresholds[emotion]).astype(int)
val_macro_f1 = f1_score(
    val_labels,
    optimized_val_preds,
    average="macro",
    zero_division=0
)
print("\nValidation Macro F1:")
print(f"{val_macro_f1:.4f}")



Validation Macro F1:
0.8458


#Review detailed validation metrics
This section reports overall validation Macro F1 and per-emotion precision, recall, and F1 using the tuned thresholds.
The per-label metrics help identify which emotions are predicted more or less effectively. The selected thresholds and these validation metrics can be documented in the project README after checking the actual output produced by the notebook.

In [ ]:
val_macro_f1 = f1_score(
    val_labels,
    optimized_val_preds,
    average="macro",
    zero_division=0
)

precision = precision_score(
    val_labels,
    optimized_val_preds,
    average=None,
    zero_division=0
)

recall = recall_score(
    val_labels,
    optimized_val_preds,
    average=None,
    zero_division=0
)

f1 = f1_score(
    val_labels,
    optimized_val_preds,
    average=None,
    zero_division=0
)


print("\n" + "=" * 75)
print("VALIDATION RESULTS")
print("=" * 75)

print(f"\nOverall Macro F1: {val_macro_f1:.4f}")

print("\nEmotion-wise Performance:")
print("-" * 75)

for i, emotion in enumerate(emotion_names):

    print(
        f"{emotion:10s} | "
        f"Threshold: {best_thresholds[emotion]:.2f} | "
        f"Precision: {precision[i]:.4f} | "
        f"Recall: {recall[i]:.4f} | "
        f"F1: {f1[i]:.4f}"
    )

print("-" * 75)


VALIDATION RESULTS

Overall Macro F1: 0.8458

Emotion-wise Performance:
---------------------------------------------------------------------------
anger      | Threshold: 0.32 | Precision: 0.8345 | Recall: 0.7516 | F1: 0.7908
joy        | Threshold: 0.27 | Precision: 0.8187 | Recall: 0.8723 | F1: 0.8446
fear       | Threshold: 0.73 | Precision: 0.9060 | Recall: 0.8773 | F1: 0.8914
sadness    | Threshold: 0.22 | Precision: 0.8337 | Recall: 0.8840 | F1: 0.8581
surprise   | Threshold: 0.46 | Precision: 0.8071 | Recall: 0.8840 | F1: 0.8438
---------------------------------------------------------------------------


#Load the saved checkpoint and prepare the test dataset
This cell starts the inference workflow:
1. Recreates the same EmotionClassifier architecture.
2. Loads the saved weights from best_model.pth.
3. Moves the model to the available device and switches it to evaluation mode.
4. Defines a TestDataset that tokenizes test text with the same maximum length of 256.
5. Creates a test DataLoader with batches of 16 and no shuffling.
The checkpoint file must exist at the path used by torch.load. Ensure the architecture and tokenizer settings match those used when the checkpoint was saved.

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EmotionClassifier(
    model_name="microsoft/deberta-v3-base",
    num_labels=5,
    dropout=0.3
)
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print("Best DeBERTa model loaded successfully!")

class TestDataset(Dataset):

    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["text"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = self.texts[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

test_ds = TestDataset(test,tokenizer,max_len=256)
test_loader = DataLoader(test_ds,batch_size=16,shuffle=False)



test_probs = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = model(input_ids=input_ids,attention_mask=attention_mask)
        probs = torch.sigmoid(logits)
        test_probs.extend(probs.cpu().numpy())


test_probs = np.array(test_probs)
print("\nTest probability shape:", test_probs.shape)
print("\nFirst prediction probabilities:")
print(test_probs[0])

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Best DeBERTa model loaded successfully!


Predicting:   0%|          | 0/107 [00:00<?, ?it/s]


Test probability shape: (1707, 5)

First prediction probabilities:
[0.9296402  0.00164251 0.99010473 0.95202595 0.01037075]


#Conclusion
This notebook demonstrates an end-to-end multi-label emotion-classification workflow with a pretrained DeBERTa-v3 backbone and PyTorch.
The pipeline covers data splitting, tokenization, batching, model fine-tuning, validation evaluation, per-emotion threshold search, and test-set submission generation.